In [1]:
from __future__ import annotations

import os

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN
import torch.nn as nn
from torch_geometric.loader import DataLoader

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset

In [2]:
model_global = torch.load("../models/gnn_checkpoint.pt", map_location=torch.device('cpu'))
model_cliff = torch.load("../models/gnn_model_clifford.pt", map_location=torch.device('cpu'))
model_haar = torch.load("../models/gnn_model_haar.pt", map_location=torch.device('cpu'))
model_quan = torch.load("../models/gnn_model_quansistor.pt", map_location=torch.device('cpu'))
model_rand = torch.load("../models/gnn_model_random.pt", map_location=torch.device('cpu'))

C:\Users\Victor\AppData\Local\Temp\ipykernel_187708\1628187572.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_global = torch.load("../models/gnn_checkpoint.pt", m

In [3]:
model_global

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0005,  0.0235, -0.0373,  ...,  0.0409,  0.0870, -0.2433],
                       [ 0.0617, -0.0206, -0.2062,  ...,  0.1419, -0.1107,  0.0150],
                       [-0.1005,  0.1833, -0.0209,  ...,  0.2241,  0.0860, -0.2587],
                       ...,
                       [-0.0251,  0.1399, -0.3376,  ...,  0.1401, -0.1835, -0.0181],
                       [-0.1884,  0.0726,  0.1752,  ...,  0.0680,  0.0324,  0.1162],
                       [-0.1461, -0.1813,  0.0248,  ...,  0.1236, -0.1774,  0.3059]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.0870, -0.0908, -0.1373, -0.1195, -0.0484,  0.1685, -0.0261,  0.1812,
                        0.1959,  0.0181, -0.1729,  0.1409,  0.0154, -0.1378,  0.1569, -0.0627,
                       -0.0193, -0.0368,  0.0887, -0.1961,  0.1848, -0.1056,  0.1888, -0.2046,
                        0.0560, -0.1984, -0.0283, -0.1532, -0.

In [4]:
model_cliff

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.1953, -0.0894, -0.1445,  ..., -0.1819, -0.0232,  0.0893],
                       [ 0.0628, -0.0125, -0.2814,  ..., -0.0698, -0.1099,  0.1851],
                       [-0.1593,  0.0672, -0.0689,  ..., -0.1042, -0.0139, -0.0118],
                       ...,
                       [ 0.0318,  0.0033,  0.1739,  ..., -0.1510,  0.1792, -0.0794],
                       [ 0.1875,  0.0555,  0.4490,  ...,  0.1627, -0.1014, -0.1114],
                       [ 0.0150, -0.1242, -0.1166,  ..., -0.0039,  0.1290, -0.1665]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.1918, -0.0047,  0.1410,  0.1112, -0.1588, -0.2048, -0.0874, -0.1378,
                        0.0293,  0.1833, -0.1183,  0.0446,  0.0981,  0.0717, -0.0321, -0.0955,
                        0.0824, -0.1532,  0.1248, -0.0613, -0.0542, -0.1567,  0.1149,  0.1057,
                       -0.0086, -0.1841,  0.1168, -0.1778, -0.

In [5]:
model_quan

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-8.4863e-02, -2.5144e-02,  6.4400e-02,  ..., -1.8593e-01,
                        -1.0805e-02,  1.8030e-03],
                       [-3.4380e-02,  1.1733e-01,  1.9030e-01,  ..., -1.9169e-01,
                         1.7281e-01,  1.3009e-01],
                       [ 1.1439e-01,  3.3987e-02,  2.8016e-02,  ...,  1.6253e-01,
                         5.5502e-02, -9.3156e-03],
                       ...,
                       [ 6.7760e-02, -1.9905e-01, -1.0294e-01,  ..., -3.2754e-02,
                         2.5076e-01,  1.0737e-01],
                       [-3.9010e-02,  5.4049e-02,  1.3753e-01,  ..., -8.2912e-05,
                        -9.5354e-02,  9.9133e-02],
                       [-1.5016e-01,  4.8966e-02, -2.2507e-02,  ..., -5.7712e-02,
                         1.6697e-01, -1.4497e-01]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.1228,  0.1170,  0.1502, -0.0388, 

In [6]:
model_rand

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0007, -0.1533,  0.0732,  ..., -0.1604, -0.0793,  0.1465],
                       [-0.1464, -0.0406,  0.0236,  ...,  0.0741, -0.0900, -0.0766],
                       [-0.0202,  0.0146, -0.0091,  ..., -0.0801,  0.1551,  0.0296],
                       ...,
                       [ 0.0401, -0.0700, -0.1986,  ..., -0.1893, -0.0378, -0.0794],
                       [ 0.0599, -0.0324,  0.1372,  ..., -0.0159, -0.1059, -0.0457],
                       [ 0.1192, -0.1155, -0.0934,  ..., -0.0276, -0.0770, -0.1593]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.0072,  0.0521, -0.0213,  0.0785,  0.1335, -0.0202, -0.0026, -0.0168,
                        0.1276, -0.1665, -0.1652,  0.0590, -0.0083, -0.0433,  0.0034,  0.1181,
                        0.1372, -0.0929,  0.0422, -0.1801, -0.1011, -0.1426,  0.0688,  0.1828,
                       -0.1806,  0.0860, -0.0396,  0.1646, -0.

In [7]:
model_haar

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0075, -0.1611,  0.1291,  ..., -0.0183,  0.0736,  0.0763],
                       [-0.2033, -0.0041,  0.1303,  ..., -0.0209,  0.0185,  0.1235],
                       [ 0.0593,  0.1237,  0.1218,  ...,  0.0478, -0.1415, -0.1108],
                       ...,
                       [-0.1061,  0.1119,  0.1158,  ..., -0.0742, -0.2059, -0.0482],
                       [-0.1047,  0.0186,  0.2032,  ...,  0.1702, -0.0210,  0.2195],
                       [-0.0380, -0.1979,  0.1559,  ..., -0.1139, -0.0618, -0.1758]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.1152,  0.1069,  0.0338,  0.0692, -0.1860,  0.1838, -0.0412, -0.0103,
                        0.1553,  0.0855,  0.1636, -0.1471,  0.0137, -0.1542,  0.1631,  0.0427,
                       -0.0796,  0.1785, -0.0317, -0.0553,  0.0770,  0.1096, -0.0145, -0.1576,
                       -0.0764, -0.0507,  0.0125,  0.0600,  0.

In [8]:
def collect_pred_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    pred_root = d / "predictions"

    if family is not None:
        paths = sorted((pred_root / family).glob("*.pt"))
    else:
        paths = []
        if pred_root.exists():
            for family_dir in sorted(pred_root.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))

    if not paths:
        paths = sorted(d.glob("*.pt"))

    return [str(p.resolve()) for p in paths]


def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    # Include full resolved paths in the digest to avoid collisions across folders/families.
    canonical = "|".join(sorted(str(Path(p).resolve()) for p in paths))
    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]
    tag = f"_{suffix}" if suffix else ""
    cache_dir = Path("..") / "qqe" / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    return str(cache_dir.resolve())

In [9]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [10]:
def build_pred_loaders_two_stage(
    pt_paths: list[str],
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    pred_ds = padded_dataset
    pin_mem = torch.cuda.is_available()
    return (
        dataset,
        padded_dataset,
        DataLoader(pred_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [67]:
family = "haar"
global_feature_variant = "binned"
node_feature_backend_variant = None

In [82]:
MODEL_STATE_PATH = f"../models/gnn_model_{family}.pt"
CHECKPOINT_PATH = f"../models/gnn_model_{family}.pt"

In [83]:
pred_data_paths = collect_pred_paths("../outputs/data", family=family)
print(f"Collected {len(pred_data_paths)} .pt files for dataset.")

Collected 8750 .pt files for dataset.


In [84]:
dataset, padded_data, pred_loader, node_in_dim, global_in_dim = build_pred_loaders_two_stage(
    pred_data_paths,  # Use first 10 paths for demonstration
    batch_size=32,
)

In [92]:
dataset[0]

Data(
  x=[85, 25],
  edge_index=[2, 134],
  y=[1],
  global_features=[1, 3],
  num_qubits=12,
  gate_counts={ haar_count=61 },
  meta={
    cid='haar_Q12_L11_S1220274776',
    family='haar',
    n_qubits=12,
    n_layers=11,
    seed=1220274776,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [85]:
# --------------------------------------------------
# 1. Build prediction dataset with training-compatible feature config
# --------------------------------------------------
if not pred_data_paths:
    raise RuntimeError("No prediction .pt files found. Check outputs/data/predictions/<family>.")

checkpoint = None
model_config = {}
fixed_all_gate_keys = None

if Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    if isinstance(checkpoint, dict):
        model_config = checkpoint.get("model_config", {}) or {}
        feature_config = checkpoint.get("feature_config", {}) or {}
        fixed_all_gate_keys = feature_config.get("all_gate_keys", None)

suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
root = _cache_root_for_paths(pred_data_paths, suffix=suffix)

pred_dataset = QuantumCircuitGraphDataset(
    root=root,
    pt_paths=pred_data_paths,
    global_feature_variant=global_feature_variant,
    node_feature_backend_variant=node_feature_backend_variant,
    fixed_all_gate_keys=fixed_all_gate_keys,
)

trained_node_in_dim = model_config.get("node_in_dim", None)
trained_global_in_dim = model_config.get("global_in_dim", None)

padded_pred_dataset = PaddedGraphDatasetWrapper(
    pred_dataset,
    target_node_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
    target_global_dim=trained_global_in_dim if trained_global_in_dim is not None else None,
    target_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
 )

pred_loader = DataLoader(
    padded_pred_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

node_in_dim = padded_pred_dataset.target_dim
global_in_dim = padded_pred_dataset.target_global_dim

print(f"Prediction graphs: {len(pred_dataset)}")
print(f"Prediction node feature dim: {node_in_dim}")
print(f"Prediction global feature dim: {global_in_dim}")
if trained_node_in_dim is not None or trained_global_in_dim is not None:
    print(f"Trained node/global dims from checkpoint: {trained_node_in_dim}/{trained_global_in_dim}")

Prediction graphs: 8750
Prediction node feature dim: 23
Prediction global feature dim: 3
Trained node/global dims from checkpoint: 23/3


C:\Users\Victor\AppData\Local\Temp\ipykernel_187708\351848649.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu

In [86]:
pred_dataset[0]

Data(
  x=[85, 25],
  edge_index=[2, 134],
  y=[1],
  global_features=[1, 3],
  num_qubits=12,
  gate_counts={ haar_count=61 },
  meta={
    cid='haar_Q12_L11_S1220274776',
    family='haar',
    n_qubits=12,
    n_layers=11,
    seed=1220274776,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [87]:
# --------------------------------------------------
# 2. Validate dataset/model dimensions
# --------------------------------------------------
print("node_in_dim (prediction) =", node_in_dim)
print("global_in_dim (prediction) =", global_in_dim)
print("node_in_dim (trained) =", trained_node_in_dim)
print("global_in_dim (trained) =", trained_global_in_dim)

if trained_node_in_dim is not None and node_in_dim != trained_node_in_dim:
    print(f"Node dim will be adapted to trained dim {trained_node_in_dim} at inference time.")

if trained_global_in_dim is not None and global_in_dim != trained_global_in_dim:
    print(f"Global feature dim will be adapted to trained dim {trained_global_in_dim} at inference time.")

node_in_dim (prediction) = 23
global_in_dim (prediction) = 3
node_in_dim (trained) = 23
global_in_dim (trained) = 3


In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [89]:
# --------------------------------------------------
# 3. Rebuild model and load weights robustly
# --------------------------------------------------
def _extract_state_dict(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict) and "model_state_dict" in payload:
        return payload["model_state_dict"]
    if isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    raise RuntimeError("Unsupported model file format.")

if checkpoint is not None and isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    if not Path(MODEL_STATE_PATH).exists():
        raise FileNotFoundError(f"Could not find model weights at {MODEL_STATE_PATH}")
    raw_payload = torch.load(MODEL_STATE_PATH, map_location="cpu")
    state_dict = _extract_state_dict(raw_payload)

model_kwargs = {
    "node_in_dim": int(trained_node_in_dim if trained_node_in_dim is not None else node_in_dim),
    "global_in_dim": int(trained_global_in_dim if trained_global_in_dim is not None else global_in_dim),
    "gnn_hidden": int(model_config.get("gnn_hidden", 32)),
    "gnn_heads": int(model_config.get("gnn_heads", 8)),
    "global_hidden": int(model_config.get("global_hidden", 16)),
    "reg_hidden": int(model_config.get("reg_hidden", 16)),
    "num_layers": int(model_config.get("num_layers", 5)),
    "dropout_rate": float(model_config.get("dropout_rate", 0.1)),
}

model = GNN(**model_kwargs).to(device)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
model.eval()

print("Loaded model config:", model_kwargs)
if missing_keys:
    print("Missing keys:", missing_keys)
if unexpected_keys:
    print("Unexpected keys:", unexpected_keys)

Loaded model config: {'node_in_dim': 23, 'global_in_dim': 3, 'gnn_hidden': 32, 'gnn_heads': 8, 'global_hidden': 16, 'reg_hidden': 16, 'num_layers': 5, 'dropout_rate': 0.1}


In [90]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [7.202025413513184, 7.202025890350342, 7.202024936676025, 7.202025413513184, 7.202025413513184, 7.202025413513184, 7.202025413513184, 7.202024936676025, 7.202025413513184, 7.202025890350342]
Total predictions: 8750


In [77]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [7.105358600616455, 7.105358600616455, 7.105358600616455, 7.105359077453613, 7.105358600616455, 7.105358600616455, 7.105358123779297, 7.105358600616455, 7.105358600616455, 7.105358600616455]
Total predictions: 8750


In [91]:
predictions

[7.202025413513184,
 7.202025890350342,
 7.202024936676025,
 7.202025413513184,
 7.202025413513184,
 7.202025413513184,
 7.202025413513184,
 7.202024936676025,
 7.202025413513184,
 7.202025890350342,
 7.202024936676025,
 7.202025890350342,
 7.202025890350342,
 7.202025413513184,
 7.202025413513184,
 7.202025413513184,
 7.202024936676025,
 7.202025890350342,
 7.202025413513184,
 7.202025413513184,
 7.202025890350342,
 7.202025413513184,
 7.202025413513184,
 7.202025890350342,
 7.202025413513184,
 7.340615749359131,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.340615749359131,
 7.340615272521973,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.340615749359131,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,
 7.3406147956848145,


In [78]:
def collect_pt_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "encoding_data_pennylane" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [79]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [80]:
data_paths = collect_pt_paths("../outputs/data", family=None)
print(f"Collected {len(data_paths)} .pt files for dataset.")

Collected 40000 .pt files for dataset.
